# 06.06_All_h5ad_to_mtx_Python

OG AnnData 转 Matrix Market。

- 当前文件：`analysis/06_single_cell_analysis/06.06_All_h5ad_to_mtx_Python.ipynb`
- 原始来源：`Codes/06.06_h5ad_to_mtx.ipynb`（旧编号仅用于溯源）。
- 运行内核：**python**。
- 导入依赖：`anndata`, `os`, `pandas`, `scipy.io`, `scipy.sparse`。
- 当前编号与流程见 `docs/workflow.md`、`docs/code_index.md`。
- 仅更新整理版导读；原分析单元格、参数和顺序保持不变。原始 cell 索引在本文件中加 1。


## Auco 测试

In [ ]:
import anndata
import pandas as pd
from scipy.io import mmwrite
from scipy.sparse import csr_matrix, issparse

# 加载.h5ad文件
adata = anndata.read_h5ad('/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/scOrthoGeneH5ad/Auco.OG.normalized.h5ad')
print(adata)

# 检查adata.X是否是稀疏矩阵
if issparse(adata.X):
    print("adata.X 是一个稀疏矩阵。")
else:
    print("adata.X 不是一个稀疏矩阵。")
    # 如果adata.X不是稀疏矩阵，将其转换为稀疏矩阵
    adata.X = csr_matrix(adata.X)

# 将稀疏矩阵保存为MTX文件
mmwrite('/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/scOrthoGeneRDS/mtx_obs_var_umap/Auco.OG.matrix.mtx', adata.X)

# 转换obs和var为csv
adata.obs.to_csv('/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/scOrthoGeneRDS/mtx_obs_var_umap/Auco.OG.obs.csv')
adata.var.to_csv('/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/scOrthoGeneRDS/mtx_obs_var_umap/Auco.OG.var.csv')

# 保存 X_umap
if 'X_umap' in adata.obsm:
    pd.DataFrame(
        adata.obsm['X_umap'],
        index=adata.obs_names,
        columns=['UMAP_1', 'UMAP_2'] if adata.obsm['X_umap'].shape[1] == 2 else [f'UMAP_{i+1}' for i in range(adata.obsm['X_umap'].shape[1])]
    ).to_csv('/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/scOrthoGeneRDS/mtx_obs_var_umap/Auco.OG.X_umap.csv')

## 批量处理

In [ ]:
import anndata
import pandas as pd
from scipy.io import mmwrite
from scipy.sparse import csr_matrix, issparse
import os

def export_mtx_obs_var_umap(
    h5ad_path,
    outdir,
    prefix=None
):
    # 文件名前缀
    if prefix is None:
        prefix = os.path.splitext(os.path.basename(h5ad_path))[0]
    print(f"处理 {h5ad_path}")
    adata = anndata.read_h5ad(h5ad_path)
    print(adata)
    # 稀疏检查
    if not issparse(adata.X):
        adata.X = csr_matrix(adata.X)
    # 写 MTX
    mmwrite(os.path.join(outdir, f"{prefix}.matrix.mtx"), adata.X)
    # 写 obs/var
    adata.obs.to_csv(os.path.join(outdir, f"{prefix}.obs.csv"))
    adata.var.to_csv(os.path.join(outdir, f"{prefix}.var.csv"))
    # 写 X_umap
    if 'X_umap' in adata.obsm:
        umap = pd.DataFrame(
            adata.obsm['X_umap'],
            index=adata.obs_names,
            columns=['UMAP_1', 'UMAP_2'] if adata.obsm['X_umap'].shape[1] == 2 else [f'UMAP_{i+1}' for i in range(adata.obsm['X_umap'].shape[1])]
        )
        umap.to_csv(os.path.join(outdir, f"{prefix}.X_umap.csv"))
    print(f"已导出: {prefix}")

# ====== 用法示例 ======
species = ['Dare', 'Neve', 'Clhe', 'TrH1', 'TrH2', 'HoH13', 'ClH23', 'Spla']

indir = '/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/scOrthoGeneH5ad/'
outdir = '/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/scOrthoGeneRDS/mtx_obs_var_umap/'

for sp in species:
    h5ad_path = os.path.join(indir, f"{sp}.OG.normalized.h5ad")
    export_mtx_obs_var_umap(h5ad_path, outdir, f"{sp}.OG")
